# 04 – Shipping Cost Model
Orders are small parcels, so they are assumed to be shipped by an external parcel carrier.
Shipping cost is estimated with our own formula of weight and distance, calibrated on two real carrier rate tables.

**Key definitions:**
- **Order** = the first part of `order_id` (e.g. `US-2021-128055`). The raw `order_id` is order + product code, so one order can contain several products (verified below).
- **Parcel** = order × factory: products from the same factory in one order ship together.
- **Customer delivery fee** = flat fee per order, equal to the average shipping cost per order in the current model (separately for each rate table). The fee is the same in every scenario, so dollar savings do not depend on it.
- **Margin after delivery** = gross profit + delivery fee − carrier cost (as % of revenue = sales + fee).

**Scenarios:**
- **current** – each product ships from its assigned factory
- **realistic** – each product ships from the nearest factory (by road) that already makes products of the same division
- **optimum** – each order ships from the nearest of all five factories (theoretical upper bound)

**Assumptions:**
- Weight: 2 oz per unit + 0.5 lb packaging per parcel; billable weight rounded up to the next full pound (carrier rule)
- Distance for pricing: straight-line miles (carrier zones are radius-based); factory choice: road miles (OpenRouteService, driving-hgv)
- Rate tables: **base** = USPS Ground Advantage commercial (Notice 123, effective July 12, 2026), **high** = UPS Ground published daily rates (effective Dec 22, 2025)
- Base rates only: fuel, residential and delivery-area surcharges are excluded

**How to read the results:** dollar amounts are illustrative; the reliable results are proportions and miles.
The model can be re-run with the company's real shipping costs.

## Price formulas
Each rate table is fitted with: price = a + b·weight + c·miles + e·weight·miles

In [2]:
import numpy as np
import pandas as pd

# Midpoints of zone mileage bands, zones 1–8 (zone 8 = 1801+ miles → 2100)
ZONE_MILES = [25, 100, 225, 450, 800, 1200, 1600, 2100]

RATE_CARDS = {
    # USPS Ground Advantage commercial, Notice 123, effective July 12, 2026 (zones 1–8)
    "usps_commercial": {
        "zones": ZONE_MILES,
        "rates": {
            1: [7.61, 7.68, 8.00, 8.15, 8.74, 9.63, 9.98, 10.67],
            2: [7.99, 8.08, 8.26, 8.51, 9.95, 11.58, 12.00, 12.87],
            3: [8.64, 8.66, 9.14, 9.67, 11.57, 13.59, 14.36, 15.75],
            4: [9.28, 9.34, 9.70, 10.65, 12.84, 15.16, 16.19, 18.01],
            5: [9.70, 9.76, 10.14, 11.02, 13.48, 15.89, 17.12, 19.19],
        },
    },
    # UPS Ground published daily rates, effective Dec 22, 2025 (zones 2–8, no zone 1)
    "ups_ground": {
        "zones": ZONE_MILES[1:],
        "rates": {
            1: [11.99, 12.38, 13.51, 14.10, 14.60, 14.83, 15.03],
            2: [12.85, 14.15, 15.46, 15.79, 16.49, 17.18, 17.40],
            3: [13.38, 14.91, 16.08, 16.93, 17.65, 18.28, 19.14],
            5: [14.19, 15.71, 17.15, 18.65, 19.45, 20.54, 21.72],
        },
    },
}

def calibrate(card):
    """Fit price = a + b*weight + c*miles + e*weight*miles to a carrier rate card."""
    rows = [(w, d, p) for w, prices in card["rates"].items() for d, p in zip(card["zones"], prices)]
    t = pd.DataFrame(rows, columns=["weight_lb", "miles", "price"])
    X = np.column_stack([np.ones(len(t)), t["weight_lb"], t["miles"], t["weight_lb"] * t["miles"]])
    coef, *_ = np.linalg.lstsq(X, t["price"], rcond=None)
    pred = X @ coef
    r2 = 1 - ((t["price"] - pred) ** 2).sum() / ((t["price"] - t["price"].mean()) ** 2).sum()
    return {"a": coef[0], "b": coef[1], "c": coef[2], "e": coef[3],
            "r2": r2, "max_error_usd": (t["price"] - pred).abs().max()}

formulas = pd.DataFrame({name: calibrate(card) for name, card in RATE_CARDS.items()}).T
display(formulas.round(5))

formulas.to_csv("../data/processed/cost_formulas.csv", index_label="variant")

,a,b,c,e,r2,max_error_usd
usps_commercial,7.02161,0.46895,0.00084,0.00086,0.98256,1.11336
ups_ground,12.18854,0.60382,0.00102,0.00049,0.92457,1.36739


## Orders and parcels
The raw `order_id` combines the order number and the product code (`US-2021-128055-CHO-TRI-54000`).
Hypothesis: the real order is the first part (`order_no`). Check: all lines of one `order_no` should share
the customer, ZIP code, order date and ship mode.

In [3]:
P = "../data/processed/"
sales = pd.read_csv(P + "sales.csv", dtype={"postal_code": str})
products = pd.read_csv(P + "products.csv")

lines = sales.merge(products[["product_id", "factory"]], on="product_id")
lines["order_no"] = lines["order_id"].str.rsplit("-", n=3).str[0]
display(lines[["order_id", "order_no", "product_id"]].head(3))

check = lines.groupby("order_no").agg(
    order_ids=("order_id", "nunique"),
    customers=("customer_id", "nunique"),
    zips=("postal_code", "nunique"),
    dates=("order_date", "nunique"),
    modes=("ship_mode", "nunique"),
    factories=("factory", "nunique"),
)
multi = check[check["order_ids"] > 1]
print("Order IDs:", lines["order_id"].nunique(), "| Orders (order_no):", len(check),
      "| Customers:", lines["customer_id"].nunique())
print("Orders with several order IDs:", len(multi))
print("...of which with one customer, ZIP, date and ship mode:",
      ((multi[["customers", "zips", "dates", "modes"]] == 1).all(axis=1)).sum())
print("Orders per customer (max):", lines.groupby("customer_id")["order_no"].nunique().max())
print("Factories per order:", check["factories"].value_counts().sort_index().to_dict())

,order_id,order_no,product_id
0,US-2021-128055-CHO-TRI-54000,US-2021-128055,CHO-TRI-54000
1,US-2021-128055-CHO-SCR-58000,US-2021-128055,CHO-SCR-58000
2,US-2021-138100-CHO-FUD-51000,US-2021-138100,CHO-FUD-51000


Order IDs: 8389 | Orders (order_no): 5009 | Customers: 5009
Orders with several order IDs: 2207
...of which with one customer, ZIP, date and ship mode: 2207
Orders per customer (max): 1
Factories per order: {1: 3320, 2: 1570, 3: 111, 4: 8}


**Findings:**
- The hypothesis holds: all 2,207 orders with several `order_id`s have one customer, ZIP code, date and ship mode → `order_no` is the real order.
- 5,009 orders = 5,009 customers: every customer ordered exactly once in four years (synthetic data → no repeat-customer analysis possible).
- 1,689 orders (34%) contain products from 2–4 factories → in the current model they ship as several parcels for one delivery fee.

**Sanity check:** could `order_no` match by coincidence, e.g. a counter reused by different customers? No – within every `order_no` the customer, ZIP, date and ship mode are identical, and each customer has exactly one `order_no`.
Earlier versions of this project treated `order_id` as an order; the check "one factory per order" in notebook 03 was true only because each `order_id` holds a single product.
→ Correction: parcel = order × factory.

## Scenarios
Each order line is assigned a factory in every scenario. Lines of the same order that end up in the same factory will ship together (next section).

In [4]:
factories = pd.read_csv(P + "factories.csv")
zips = pd.read_csv(P + "zips.csv", dtype={"zip": str})
road = pd.read_csv(P + "road_distances.csv", dtype={"zip": str})

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 3958.8 * 2 * np.arcsin(np.sqrt(a))

# Distance tables: rows = ZIP codes, columns = factories
road_wide = road.pivot(index="zip", columns="factory", values="road_miles")
pairs = zips.merge(factories, how="cross")
pairs["straight_miles"] = haversine_miles(pairs["lat"], pairs["lng"], pairs["latitude"], pairs["longitude"])
straight_wide = pairs.pivot(index="zip", columns="factory", values="straight_miles")

division_factories = products.groupby("division")["factory"].unique().apply(list).to_dict()
print("Factories per division:", division_factories)

lines["current_factory"] = lines["factory"]
lines["realistic_factory"] = [road_wide.loc[z, division_factories[d]].idxmin()
                              for z, d in zip(lines["postal_code"], lines["division"])]
lines["optimum_factory"] = road_wide.loc[lines["postal_code"]].idxmin(axis=1).values

SCENARIOS = ["current", "realistic", "optimum"]
for sc in SCENARIOS:
    lines[f"{sc}_road_miles"] = [road_wide.at[z, f] for z, f in zip(lines["postal_code"], lines[f"{sc}_factory"])]

# A closer factory can never be farther away
assert (lines["realistic_road_miles"] <= lines["current_road_miles"] + 1e-9).all()
assert (lines["optimum_road_miles"] <= lines["realistic_road_miles"] + 1e-9).all()

for sc in ["realistic", "optimum"]:
    moved = lines[f"{sc}_factory"] != lines["current_factory"]
    print(f"{sc}: {moved.sum()} of {len(lines)} order lines change factory ({moved.mean() * 100:.1f}%)")

display(pd.crosstab(lines["current_factory"], lines["realistic_factory"]))

Factories per division: {'Chocolate': ["Lot's O' Nuts", "Wicked Choccy's"], 'Other': ['Sugar Shack', 'Secret Factory', 'The Other Factory'], 'Sugar': ['Sugar Shack', 'Secret Factory', 'The Other Factory']}
realistic: 5256 of 9994 order lines change factory (52.6%)
optimum: 7074 of 9994 order lines change factory (70.8%)


realistic_factory,Lot's O' Nuts,Secret Factory,Sugar Shack,The Other Factory,Wicked Choccy's
current_factory,,,,,
Lot's O' Nuts,1978,0,0,0,3610
Secret Factory,0,111,25,77,0
Sugar Shack,0,23,2,8,0
The Other Factory,0,44,17,39,0
Wicked Choccy's,1452,0,0,0,2608


**Findings:**
- Realistic: 5,256 of 9,994 order lines (52.6%) change factory; optimum: 7,074 (70.8%).
- Chocolate dominates the changes: 3,610 lines move Lot's O' Nuts → Wicked Choccy's and 1,452 the other way.
  Wicked Choccy's would handle 6,218 lines instead of 4,060 (+53%), Lot's O' Nuts 3,430 instead of 5,588 (−39%).
- Other and Sugar (345 lines) are reshuffled between the three non-chocolate factories; volumes are small.

**Sanity check:** is it plausible that over half of the lines are shipped from the "wrong" factory today?
Yes – the two chocolate factories sit at opposite ends of the country (Arizona and Georgia) and each makes different bars,
while customers are spread across the whole US, so roughly half of the bars travel coast to coast.
Both `assert`s passed: no line got a longer route in any scenario.
Caveat: the realistic scenario requires both chocolate factories to make all five bars → capacity must be verified (limitation).

## Parcels and shipping cost
Lines of the same order assigned to the same factory form one parcel. Cost is calculated per parcel for both rate tables.

In [5]:
def shipping_cost(weight, miles, f):
    return f["a"] + f["b"] * weight + f["c"] * miles + f["e"] * weight * miles

# One row per scenario × parcel (order × factory)
parcels = []
for sc in SCENARIOS:
    p = (lines.groupby(["order_no", f"{sc}_factory"], as_index=False)
              .agg(postal_code=("postal_code", "first"), units=("units", "sum"))
              .rename(columns={f"{sc}_factory": "factory"}))
    p.insert(0, "scenario", sc)
    parcels.append(p)
parcels = pd.concat(parcels, ignore_index=True)

parcels["weight_lb"] = parcels["units"] * 2 / 16 + 0.5
parcels["billable_lb"] = np.ceil(parcels["weight_lb"])
parcels["road_miles"] = [road_wide.at[z, f] for z, f in zip(parcels["postal_code"], parcels["factory"])]
parcels["straight_miles"] = [straight_wide.at[z, f] for z, f in zip(parcels["postal_code"], parcels["factory"])]
for variant, f in formulas.iterrows():
    parcels[f"cost_{variant}"] = shipping_cost(parcels["billable_lb"], parcels["straight_miles"], f)
    parcels[f"fixed_{variant}"] = f["a"] + f["b"] * parcels["billable_lb"]   # distance-independent part

summary = parcels.groupby("scenario").agg(
    parcels=("order_no", "size"),
    road_miles=("road_miles", "sum"),
    **{f"cost_{v}": (f"cost_{v}", "sum") for v in formulas.index},
).loc[SCENARIOS]
savings_pct = (1 - summary / summary.loc["current"]) * 100
display(summary.round(0))
display(savings_pct.round(1))

# Where do the savings come from: fewer parcels (fixed part) or fewer miles (distance part)?
split = []
for v in formulas.index:
    t = parcels.groupby("scenario")[[f"cost_{v}", f"fixed_{v}"]].sum().loc[SCENARIOS]
    t["distance"] = t[f"cost_{v}"] - t[f"fixed_{v}"]
    saved = t.loc["current"] - t
    for sc in ["realistic", "optimum"]:
        split.append({"variant": v, "scenario": sc,
                      "saved_usd": saved.loc[sc, f"cost_{v}"],
                      "from_fewer_parcels_pct": saved.loc[sc, f"fixed_{v}"] / saved.loc[sc, f"cost_{v}"] * 100,
                      "from_fewer_miles_pct": saved.loc[sc, "distance"] / saved.loc[sc, f"cost_{v}"] * 100})
display(pd.DataFrame(split).round(1))

,parcels,road_miles,cost_usps_commercial,cost_ups_ground
scenario,,,,
current,6825,9810011.0,71513.0,104883.0
realistic,5239,3988018.0,49299.0,76031.0
optimum,5009,2921571.0,45616.0,71502.0


,parcels,road_miles,cost_usps_commercial,cost_ups_ground
scenario,,,,
current,0.0,0.0,0.0,0.0
realistic,23.2,59.3,31.1,27.5
optimum,26.6,70.2,36.2,31.8


,variant,scenario,saved_usd,from_fewer_parcels_pct,from_fewer_miles_pct
0,usps_commercial,realistic,22213.8,52.9,47.1
1,usps_commercial,optimum,25897.5,51.9,48.1
2,ups_ground,realistic,28851.9,69.7,30.3
3,ups_ground,optimum,33381.6,69.0,31.0


**Findings:**
- Realistic: parcels −23% (6,825 → 5,239), road miles −59%, shipping cost −31% (USPS) / −28% (UPS).
  Optimum: parcels −27%, miles −70%, cost −36% / −32%. Realistic captures ~86% of the optimum's savings.
- Over half of the savings come from **fewer parcels** (53% USPS, 70% UPS), the rest from shorter routes →
  consolidating an order into one parcel is as strong a lever as distance.
- UPS savings depend more on parcel count because its fixed part per parcel is higher (a ≈ $12.19 vs $7.02).
- The previous model (parcel = `order_id`) underestimated the savings (6.8–12.0%) because it missed the consolidation effect.

**Sanity check:**
- Optimum has exactly 5,009 parcels = number of orders → every order ships as one parcel, as expected (the nearest factory depends only on the ZIP code).
- Realistic has 230 parcels more than orders – exactly the 230 orders that mix chocolate with Other/Sugar products, which are made in different factories.
- Miles fall by 59% but cost only by 31%: the distance-independent part of the price ($7–12 per parcel) does not shrink with miles → consistent with the split above.

## Delivery fee and margin after delivery
One row per order. The customer pays a flat delivery fee per order = average shipping cost per order in the current model.

In [7]:
METRICS = ["parcels", "road_miles"] + [f"cost_{v}" for v in formulas.index]

orders = lines.groupby("order_no").agg(
    customer_id=("customer_id", "first"),
    postal_code=("postal_code", "first"),
    order_date=("order_date", "first"),
    ship_mode=("ship_mode", "first"),
    units=("units", "sum"),
    sales=("sales", "sum"),
    gross_profit=("gross_profit", "sum"),
)

# Scenario results per order → wide columns, e.g. current_parcels, realistic_cost_usps_commercial
per_sc = parcels.groupby(["order_no", "scenario"]).agg(
    parcels=("factory", "size"),
    road_miles=("road_miles", "sum"),
    **{f"cost_{v}": (f"cost_{v}", "sum") for v in formulas.index},
).unstack("scenario")
per_sc.columns = [f"{sc}_{m}" for m, sc in per_sc.columns]
orders = orders.join(per_sc[[f"{sc}_{m}" for sc in SCENARIOS for m in METRICS]])

assert len(orders) == 5009
assert abs(orders["sales"].sum() - sales["sales"].sum()) < 0.01

fees = {v: orders[f"current_cost_{v}"].mean() for v in formulas.index}
print("Delivery fee per order:", {v: round(fee, 2) for v, fee in fees.items()})
print(f"Gross margin (before delivery): {orders['gross_profit'].sum() / orders['sales'].sum() * 100:.1f}%")

results = []
for v, fee in fees.items():
    for sc in SCENARIOS:
        margin = orders["gross_profit"] + fee - orders[f"{sc}_cost_{v}"]
        results.append({
            "variant": v,
            "scenario": sc,
            "margin_after_delivery_pct": margin.sum() / (orders["sales"].sum() + fee * len(orders)) * 100,
            "margin_pct_of_sales": margin.sum() / orders["sales"].sum() * 100,
            "loss_orders_pct": (margin < 0).mean() * 100,
            "cost_above_fee_pct": (orders[f"{sc}_cost_{v}"] > fee).mean() * 100,
        })
display(pd.DataFrame(results).round(1))

Delivery fee per order: {'usps_commercial': np.float64(14.28), 'ups_ground': np.float64(20.94)}
Gross margin (before delivery): 65.9%


,variant,scenario,margin_after_delivery_pct,margin_pct_of_sales,loss_orders_pct,cost_above_fee_pct
0,usps_commercial,current,43.5,65.9,0.4,35.0
1,usps_commercial,realistic,54.1,81.9,0.1,4.9
2,usps_commercial,optimum,55.8,84.6,0.0,0.1
3,ups_ground,current,37.5,65.9,1.7,33.8
4,ups_ground,realistic,49.4,86.7,0.2,4.6
5,ups_ground,optimum,51.2,90.0,0.0,0.0


**Findings:**
- Delivery fee per order: $14.28 (USPS) / $20.94 (UPS).
- Current model: margin after delivery **43.5%** (USPS) / **37.5%** (UPS).
- Only 0.4% / 1.7% of orders lose money after delivery, but **35% / 34% of orders cost more to ship than the fee** →
  customers close to the factories subsidize distant ones (cross-subsidy).
- Realistic scenario: margin after delivery **+10.6 pp** (USPS) / **+11.9 pp** (UPS); orders whose shipping exceeds the fee drop from ~35% to ~5%.
- Optimum adds only ~1.7–1.8 pp on top of realistic → the realistic scenario captures most of the benefit.

**Sanity check:**
- Margin as % of sales in the current model equals the gross margin (65.9%) exactly. This is by construction:
  the fee equals the average cost, so fee − cost sums to zero. Not a finding, a check that the formula is consistent.
- Realistic margin as % of sales (81.9% USPS) = gross margin + savings / sales = 65.9% + $22,214 / $138,830 (16.0 pp) ✓; UPS: 65.9% + 20.8 pp = 86.7% ✓.
- Why so few loss-making orders when 35% exceed the fee? Average gross profit per order is $18.27, while the average
  overrun on those orders is $7.39 (max $32.13, multi-factory orders) → gross profit almost always absorbs it.
- UPS shows a lower main margin because its higher fee enlarges the denominator (sales + fee).

In [8]:
export = orders.reset_index()
export.to_csv(P + "orders_scenarios.csv", index=False)
print("Saved:", export.shape)
print(export.columns.tolist())

Saved: (5009, 20)
['order_no', 'customer_id', 'postal_code', 'order_date', 'ship_mode', 'units', 'sales', 'gross_profit', 'current_parcels', 'current_road_miles', 'current_cost_usps_commercial', 'current_cost_ups_ground', 'realistic_parcels', 'realistic_road_miles', 'realistic_cost_usps_commercial', 'realistic_cost_ups_ground', 'optimum_parcels', 'optimum_road_miles', 'optimum_cost_usps_commercial', 'optimum_cost_ups_ground']
